# Setup

In [1]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter("traces.db")))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The agentic loop keeps calling the model until it stops by following these steps:

1.  **Initialization:** The process starts with a `while True` loop and a `has_function_calls` flag set to `False` at the beginning of each iteration.
2.  **Model Call:** Inside the loop, the model is called (`openai_client.responses.create`) with the current `messages` history and available `tools`.
3.  **Process Response:** The model's `response.output` is appended to the `messages` history. The loop then iterates through each item in the `response.output`:
    *   **Function Call:** If an item is of `type == "function_call"`, the agent executes the corresponding tool (e.g., `search`) using a helper function (`make_call`). The result of this tool call is then appended back to the `messages` history, and `has_function_calls` is set to `True`.
    *   **Message:** If an item is of `type == "message"`, it's a regular text output from the model (e.g., a final answer or a conversational turn).
4.  **Loop Te

# Question 1 & Question 2 & Question 3

In [4]:
from starter import rag
from rag_helper import RAGBase

In [5]:
class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)
        
    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            return super().search(query, num_results)
        
    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            span.set_attribute("input_tokens", usage.prompt_tokens)
            span.set_attribute("output_tokens", usage.completion_tokens)
            return response

In [6]:
traced_rag = RAGTraced(
    index = rag.index,
    llm_client = rag.llm_client,
    model = rag.model
)

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = traced_rag.rag(query)
answer

"The agentic loop keeps calling the model until it stops by using a `while` loop and a flag that checks for function calls in the model's response.\n\nHere's how it works:\n1.  **Initialize a loop:** The core of the agent is a `while True` loop that continuously interacts with the model.\n2.  **Send messages to the model:** Inside the loop, the current message history (including instructions, user question, previous model outputs, and tool results) is sent to the Large Language Model (LLM).\n3.  **Process model response:** The model generates a response. This response can contain either a final message (the answer) or a request to call a tool (like `search`).\n4.  **Check for function calls:** A flag (`has_function_calls`) is set to `True` if the model's response includes a function call.\n5.  **Execute function calls:** If there are function calls, they are executed, and their results are appended to the message history.\n6.  **Continue or break:**\n    *   If `has_function_calls` is 

In [ ]:
'''

this was the output where I got first 3 questions answers from:
{
    "name": "search",
    "context": {
        "trace_id": "0xe898d73e5b6f3650760474fb5b5506b9",
        "span_id": "0x7105b687e6ad0963",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x6cfa66f1d250a7ef",
    "start_time": "2026-07-20T20:34:06.574662Z",
    "end_time": "2026-07-20T20:34:06.574662Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "25bf93f1-c26e-4a9d-b717-c9c7badd30a5",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xe898d73e5b6f3650760474fb5b5506b9",
        "span_id": "0xde9d96fdd63a7cf2",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x6cfa66f1d250a7ef",
    "start_time": "2026-07-20T20:34:06.584297Z",
    "end_time": "2026-07-20T20:34:10.934780Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7933,
        "output_tokens": 431
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "25bf93f1-c26e-4a9d-b717-c9c7badd30a5",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag",
    "context": {
        "trace_id": "0xe898d73e5b6f3650760474fb5b5506b9",
        "span_id": "0x6cfa66f1d250a7ef",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T20:34:06.574662Z",
    "end_time": "2026-07-20T20:34:10.945105Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "25bf93f1-c26e-4a9d-b717-c9c7badd30a5",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}

'''

'\n\nthis was the output where I got first 3 questions answers from:\n\n'

- **Question 1 Answer** : 3
- **Question 2 Answer** : 7000 [closest number to 8238]
- **Question 3 Answer** : over 2000 ms [4350ms]

# Question 4

In [10]:
import sqlite3
conn = sqlite3.connect("traces.db")
print(conn.execute("SELECT DISTINCT name FROM spans").fetchall())

[('search',), ('llm',), ('rag',)]


# Question 5

In [14]:
traced_rag.rag(query)

"The agentic loop keeps calling the model until it returns a response without any function calls.\n\nHere's how it works:\n1.  **Initial Call**: The loop starts by sending the user's question and initial instructions to the LLM.\n2.  **Process Response**: The LLM's response is processed.\n    *   If the response includes a `function_call` (e.g., to `search`), the agent executes that function (e.g., `make_call`). The output of this function call is then appended to the message history. A flag `has_function_calls` is set to `True`.\n    *   If the response contains a final `message` (the answer), it's printed.\n3.  **Loop Continuation**: After processing the response, the `has_function_calls` flag is checked.\n    *   If `has_function_calls` is `True`, it means the model requested a tool call and the output has been added to the message history. The loop continues, sending the updated message history (including the tool output) back to the model for its next turn.\n    *   If `has_functi

In [12]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql("SELECT * FROM spans", conn)

df["duration_ns"] = df["end_time"] - df["start_time"]

# exclude rag, group by name, sum duration
totals = (
    df[df["name"] != "rag"]
    .groupby("name")["duration_ns"]
    .sum()
)
print(totals)

name
llm       18845461800
search       20361900
Name: duration_ns, dtype: int64


**Answer** : llm

# Question 6

In [15]:
conn = sqlite3.connect("traces.db")
df = pd.read_sql("SELECT * FROM spans", conn)

llm_tokens = df[df["name"] == "llm"]["input_tokens"]
print(llm_tokens)

1     8238.0
4     7933.0
7     7933.0
10    7933.0
13    7933.0
16    7933.0
19    7933.0
Name: input_tokens, dtype: float64


In [16]:
print("min:", llm_tokens.min())
print("max:", llm_tokens.max())
print("mean:", llm_tokens.mean())
print("% variation:", (llm_tokens.max() - llm_tokens.min()) / llm_tokens.mean() * 100)

min: 7933.0
max: 8238.0
mean: 7976.571428571428
% variation: 3.8236979726341427


**Answer** : Within 10% of each other